In [2]:
## This cell imports LangGraph, typing support, LLM wrappers, and dotenv utilities. 
## These imports prepare the notebook to build a graph workflow and read API keys from the environment.
# Stategraph : shared memory 

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

In [4]:
### This cell reads the OpenRouter API key from environment variables. 
## This keeps secret keys outside the notebook code and makes it clear which key belongs to which provider
api_key = os.getenv("OPENROUTER_API_KEY")

In [5]:
# Agent 1 - GPT-4o Mini (via OpenRouter)
openai_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 2 - Llama 3 (via OpenRouter)
huggingface_llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 3 - Qwen Coder (via OpenRouter)
groq_llm = ChatOpenAI(
    model="qwen/qwen3-coder",
    temperature=0.2,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
     max_tokens=1000
)

In [6]:
## This cell defines BlogState, the shared data structure passed between agents. It stores the user request, outline, draft, and final blog as the workflow progresses.
## This code defines the state (memory) that is shared between different nodes in a workflow, such as a LangGraph application.
## TypedDict is a special Python class used to define the structure of a dictionary.
## Supports type checking - str,int,boolen like that
class BlogState(TypedDict):
    user_request: str
    outline : str
    draft : str
    final_blog : str

In [7]:
## Outline Planner : This cell defines the first agent. It takes the user request, generates a structured outline and SEO strategy, then saves that result into the shared state as outline.
def agent_outline(state: BlogState):
    print("Agent 1 started: OpenAI outline agent")
    response = openai_llm.invoke(f"""
    You are a Senior Content Strategist and SEO Planner.

    TASK:
    Create a structured blog strategy.

    INPUT:
    {state['user_request']}

    STRICT OUTPUT FORMAT (follow exactly):

    TITLE:
    (one clear SEO-optimized title)

    INTRODUCTION:
    (2–3 sentences)

    OUTLINE:
    1.
    2.
    3.
    4.
    5.

    SEO KEYWORDS:
    - keyword1
    - keyword2
    - keyword3
    - keyword4

    CONTENT STRATEGY NOTES:
    - target audience
    - tone of writing
    - search intent
    """)

    state["outline"] = response.content.strip()
    print("Agent 1 completed")
    return state


In [8]:
### this cell defines the second agent. It takes the outline generated by the first agent, expands it into a full blog draft, and saves that result into the shared state as draft.
def agent_writer(state: BlogState):
    print("Agent 2 started: Hugging Face Writer agent")
    response = huggingface_llm.invoke(f"""
    You are a professional blog writing engine.

    INPUT STRATEGY:
    {state['outline']}

    TASK:
    Convert the strategy into a full blog post.

    STRICT RULES:
    1. Follow the outline structure exactly as provided
    2. Do NOT skip any section
    3. Do NOT change section order
    4. Use SEO keywords naturally (do not force or repeat unnecessarily)
    5. Maintain consistent professional tone
    6. Expand each section with detailed explanation
    7. Do NOT add new sections
    8. Do NOT include meta commentary

    OUTPUT:
    Return ONLY the final blog content.
    """)

    state["draft"] = response.content.strip()
    print("Agent 2 completed")
    return state

In [9]:
### This cell defines the third agent. It takes the draft generated by the second agent, reviews it for quality, and saves the final improved blog into the shared state as final_blog.
def agent_reviewer(state: BlogState):
    print("Agent 3 started: Groq agent_reviewer agent")
    response = groq_llm.invoke(f"""
    You are a Senior Editorial Reviewer, SEO Editor, and Content Quality Engineer.

    TASK:
    Improve the following blog draft and make it publication-ready.

    INPUT BLOG:
    {state['draft']}

    REVIEW CHECKLIST:
    1. Fix grammar, spelling, and readability issues
    2. Improve flow between paragraphs and sections
    3. Ensure SEO keywords are naturally used (no stuffing)
    4. Remove repetition or redundant sentences
    5. Ensure strong introduction and conclusion
    6. Maintain consistent tone (professional blog style)

    STRICT RULES:
    - Do NOT remove any major section unless it is incorrect
    - Do NOT add new sections unless necessary for clarity
    - Do NOT mention that you are reviewing

    OUTPUT FORMAT:
    Return ONLY the final improved blog. No notes, no explanation.
    """)

    state["final_blog"] = response.content.strip()
    print("Agent 3 completed")
    return state

In [10]:
## Build the Sequential LangGraph Flow
## cell creates the LangGraph state graph, adds the three agents as nodes, and connects them in order: Agent 1, then Agent 2, then Agent 3.
graph = StateGraph(BlogState)
graph.add_node("Agent 1", agent_outline)
graph.add_node("Agent 2", agent_writer)
graph.add_node("Agent 3", agent_reviewer)

## Nodes to edges
graph.add_edge(START, "Agent 1")
graph.add_edge("Agent 1", "Agent 2")
graph.add_edge("Agent 2", "Agent 3")
graph.add_edge("Agent 3", END)

In [11]:
## Compile the Workflow : This cell compiles the graph into an executable app. After compiling, the graph can accept an input request and run the full sequential workflow.
app = graph.compile()

In [12]:
user_query = "Write a blog about the benefits of AI"
result = app.invoke({"user_request": user_query})
print(result["final_blog"])

Agent 1 started: OpenAI outline agent
Agent 1 completed
Agent 2 started: Hugging Face Writer agent
Agent 2 completed
Agent 3 started: Groq agent_reviewer agent
Agent 3 completed
Unlocking the Benefits of AI: Transforming Industries and Enhancing Lives

Artificial Intelligence (AI) is revolutionizing how we live and work, delivering transformative benefits across industries. From boosting efficiency to enabling smarter decision-making, AI is reshaping business operations and personal experiences alike. In this blog, we’ll explore the key advantages of AI for both businesses and individuals.

## 1. Understanding AI: A Brief Overview

To fully appreciate the potential of Artificial Intelligence, it's important to understand what it encompasses. AI refers to the development of computer systems capable of performing tasks that typically require human intelligence—such as learning, problem-solving, and decision-making. Over time, AI has evolved from basic rule-based systems to advanced deep 

In [15]:
print(result["outline"])

TITLE:
Unlocking the Benefits of AI: Transforming Industries and Enhancing Lives

INTRODUCTION:
Artificial Intelligence (AI) is revolutionizing the way we live and work, offering unprecedented advantages across various sectors. From improving efficiency to enabling smarter decision-making, the benefits of AI are vast and transformative. In this blog, we will explore the key advantages AI brings to businesses and individuals alike.

OUTLINE:
1. Understanding AI: A Brief Overview
2. Enhanced Efficiency and Productivity in Businesses
3. Improved Decision-Making Through Data Analysis
4. AI in Healthcare: Transforming Patient Care
5. The Future of Work: How AI is Shaping Careers

SEO KEYWORDS:
- benefits of AI
- artificial intelligence advantages
- AI in business
- AI in healthcare

CONTENT STRATEGY NOTES:
- target audience: Business professionals, healthcare providers, tech enthusiasts, and general readers interested in technology.
- tone of writing: Informative yet engaging, balancing tec

In [16]:
print(result["draft"])

Unlocking the Benefits of AI: Transforming Industries and Enhancing Lives

Artificial Intelligence (AI) is revolutionizing the way we live and work, offering unprecedented advantages across various sectors. From improving efficiency to enabling smarter decision-making, the benefits of AI are vast and transformative. In this blog, we will explore the key advantages AI brings to businesses and individuals alike.

## 1. Understanding AI: A Brief Overview
To grasp the full potential of Artificial Intelligence, it's essential to understand what AI entails. AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. This technology has evolved significantly over the years, from simple rule-based systems to complex deep learning models that can analyze vast amounts of data, recognize patterns, and make predictions. The core idea behind AI is to create machines that can think and act 

In [18]:
print(result["final_blog"])

Unlocking the Benefits of AI: Transforming Industries and Enhancing Lives

Artificial Intelligence (AI) is revolutionizing how we live and work, delivering transformative benefits across industries. From boosting efficiency to enabling smarter decision-making, AI is reshaping business operations and personal experiences alike. In this blog, we’ll explore the key advantages of AI for both businesses and individuals.

## 1. Understanding AI: A Brief Overview

To fully appreciate the potential of Artificial Intelligence, it's important to understand what it encompasses. AI refers to the development of computer systems capable of performing tasks that typically require human intelligence—such as learning, problem-solving, and decision-making. Over time, AI has evolved from basic rule-based systems to advanced deep learning models that can analyze vast datasets, recognize patterns, and make predictions. At its core, AI aims to create machines that emulate human thinking and behavior, enhanc